# USDPEN Conditional Variance Model

## 1. Setup & Data

We load daily FX levels for five LatAm currency pairs (USDPEN, USDCOP, USDCLP, USDMXN, USDBRL), inspect the data for quality issues, and establish the constants used throughout the notebook.

Two market regimes are hardcoded:
- **BCRP Intervention** (Nov 2025 – Feb 2026): Peru's central bank was actively suppressing USDPEN volatility.
- **Iran Conflict** (Mar 2026+): A geopolitical shock that spiked EM FX volatility and cross-currency correlations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from IPython.display import display, HTML
import ipywidgets as widgets
import warnings
warnings.filterwarnings("ignore")

# Regime dates
INTERVENTION_START = pd.Timestamp("2025-11-01")
INTERVENTION_END = pd.Timestamp("2026-02-28")
POST_SHOCK_START = pd.Timestamp("2026-03-01")

# Model parameters
EWMA_LAMBDA = 0.94
BETA_HALFLIFE = 60
BETA_MIN_OBS = 60
SEED_WINDOW = 60
RESID_ZSCORE_WINDOW = 20
CI_LEVELS = {0.90: 1.6449, 0.95: 1.9600, 0.99: 2.5758}

# Pip = 0.0001 for USDPEN
PIP = 0.0001

# Chart style
COLORS = {"pen": "#1f77b4", "cop": "#ff7f0e", "clp": "#2ca02c", "mxn": "#d62728", "brl": "#9467bd"}
COLOR_SYS = "#3B82F6"
COLOR_IDIO = "#F97316"
COLOR_TOTAL = "#111827"
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.dpi": 100,
})

def shade_regimes(ax, alpha=0.15):
    # Add regime shading to any axis
    ylim = ax.get_ylim()
    ax.axvspan(INTERVENTION_START, INTERVENTION_END, alpha=alpha, color="#D1D5DB", label="BCRP Intervention", zorder=0)
    ax.axvspan(POST_SHOCK_START, pd.Timestamp("2026-12-31"), alpha=alpha, color="#FCA5A5", label="Iran Conflict", zorder=0)
    ax.set_ylim(ylim)

def label_regime(dt):
    if dt >= POST_SHOCK_START:
        return "post_shock"
    elif dt >= INTERVENTION_START:
        return "intervention"
    return "normal"

print("Setup complete.")

### 1.2 — Load Data

In [ ]:
df_levels = pd.read_excel("fx_latam_levels.xlsx")
print("Columns found:", df_levels.columns.tolist())
print("Shape:", df_levels.shape)
print("Dtypes:")
print(df_levels.dtypes)
display(df_levels.head())
display(df_levels.tail())

# Standardize column names
col_map = {}
for col in df_levels.columns:
    cl = col.strip().lower()
    if cl in ("date",):
        col_map[col] = "date"
    elif "pen" in cl:
        col_map[col] = "usdpen"
    elif "cop" in cl:
        col_map[col] = "usdcop"
    elif "clp" in cl:
        col_map[col] = "usdclp"
    elif "mxn" in cl:
        col_map[col] = "usdmxn"
    elif "brl" in cl:
        col_map[col] = "usdbrl"
    else:
        col_map[col] = cl

df_levels = df_levels.rename(columns=col_map)
df_levels["date"] = pd.to_datetime(df_levels["date"])

expected = {"usdpen", "usdcop", "usdclp", "usdmxn", "usdbrl"}
found = set(df_levels.columns) & expected
if found != expected:
    print(f"WARNING: Expected {expected}, found {found}. Missing: {expected - found}")

df_levels = df_levels.set_index("date").sort_index()

# Reorder columns to consistent order
df_levels = df_levels[["usdpen", "usdcop", "usdclp", "usdmxn", "usdbrl"]]

# Check for gaps > 3 business days
date_diff = pd.Series(df_levels.index).diff().dt.days
gaps = date_diff[date_diff > 5]
if len(gaps) > 0:
    print(f"\nGaps > 3 business days:")
    for idx in gaps.index:
        print(f"  {df_levels.index[idx-1].date()} -> {df_levels.index[idx].date()} ({int(date_diff.iloc[idx])} cal days)")
else:
    print("\nNo gaps > 3 business days.")

print(f"\nData loaded: {df_levels.shape[0]} rows, {df_levels.index[0].date()} to {df_levels.index[-1].date()}")
print(f"Column order: {df_levels.columns.tolist()}")

## 2. Log Returns & Diagnostics

Log returns are the natural unit for FX analysis. If S_t is the spot rate, r_t = ln(S_t / S_{t-1}). Log returns are approximately additive over time and their distribution is closer to normal than simple returns. A move from 3.6350 to 3.6400 is about +13.8 bps in log return terms, or +50 pips.

### 2.1 — Compute Returns

In [ ]:
df_returns = np.log(df_levels / df_levels.shift(1))
# Map level columns to return columns: usdpen->r_pen, usdcop->r_cop, etc.
df_returns.columns = ["r_" + c.replace("usd", "") for c in df_levels.columns]
df_returns = df_returns.dropna()

print(f"Return columns: {df_returns.columns.tolist()}")
summary = df_returns.describe().T
summary["skew"] = df_returns.skew()
summary["kurtosis"] = df_returns.kurtosis()
print("\nDaily Log Return Summary Statistics:")
display(summary.round(6))
print(f"\n{len(df_returns)} return observations")

### 2.2 — Level Plots

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)
ccy_colors = {"usdpen": COLORS["pen"], "usdclp": COLORS["clp"],
              "usdcop": COLORS["cop"], "usdbrl": COLORS["brl"], "usdmxn": COLORS["mxn"]}
for i, col in enumerate(df_levels.columns):
    ax = axes[i]
    ax.plot(df_levels.index, df_levels[col], color=ccy_colors.get(col, "black"), linewidth=1)
    current = df_levels[col].iloc[-1]
    ax.set_ylabel(col.upper())
    ax.set_title(f"{col.upper()}  (last: {current:.4f})", fontsize=11, loc="left")
    shade_regimes(ax)
axes[-1].set_xlabel("Date")
fig.suptitle("LatAm FX Levels", fontsize=13, y=1.0)
plt.tight_layout()
plt.show()

### 2.3 — Correlation Heatmap

In [ ]:
corr = df_returns.corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap="RdYlBu_r", vmin=-1, vmax=1)
labels = [c.replace("r_", "").upper() for c in corr.columns]
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.values[i,j]:.2f}", ha="center", va="center", fontsize=10,
                color="white" if abs(corr.values[i,j]) > 0.6 else "black")
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Daily Log Return Correlations (Full Sample)")
plt.tight_layout()
plt.show()

### 2.4 — Rolling 60-Day Correlation with PEN

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for reg, color_key in [("r_cop", "cop"), ("r_clp", "clp"), ("r_mxn", "mxn"), ("r_brl", "brl")]:
    rolling_corr = df_returns["r_pen"].rolling(60).corr(df_returns[reg])
    ax.plot(df_returns.index, rolling_corr, label=reg.replace("r_", "").upper(),
            color=COLORS[color_key], linewidth=1)
shade_regimes(ax)
ax.set_title("Rolling 60-Day Correlation with USDPEN")
ax.set_ylabel("Correlation")
ax.legend(loc="lower left")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## 3. Model Functions

All model functions in one cell. The model has four components:

1. **EWMA covariance matrix** — tracks how the five currencies co-move, updating daily with exponential decay (λ=0.94, ~11 day half-life).
2. **Betas** — how much USDPEN moves per unit move in each regional currency: β_i = Cov(PEN, i) / Var(i). These update daily as the covariance updates.
3. **Variance decomposition** — total PEN variance = systematic (β'Σ_region β) + idiosyncratic (residual variance). After observing the region, only the idiosyncratic piece remains.
4. **Conditional range** — expected move = β · r_region. The 95% band is ± 1.96 × √(idio_var) × spot × 10000 pips.

### 3.1 — All Functions

In [ ]:
def compute_ewma_covariance(df_rets, columns, decay=EWMA_LAMBDA, seed_window=SEED_WINDOW):
    # RiskMetrics recursive EWMA: Sigma_t = lambda * Sigma_{t-1} + (1-lambda) * r_{t-1} * r_{t-1}'
    data = df_rets[columns].values
    dates = df_rets.index
    n, k = data.shape

    # Seed with sample covariance of first seed_window obs
    sigma = np.cov(data[:seed_window], rowvar=False)

    cov_dict = {}
    var_rows = []
    corr_rows = []

    for t in range(seed_window, n):
        r_prev = data[t - 1].reshape(-1, 1)
        sigma = decay * sigma + (1 - decay) * (r_prev @ r_prev.T)
        dt = dates[t]
        cov_dict[dt] = sigma.copy()

        # Variances
        vr = {"date": dt}
        for j in range(k):
            vr[columns[j]] = sigma[j, j]
        var_rows.append(vr)

        # Correlations (unique pairs)
        stds = np.sqrt(np.diag(sigma))
        stds[stds == 0] = 1e-12
        corr_mat = sigma / np.outer(stds, stds)
        cr = {"date": dt}
        for j in range(k):
            for l in range(j + 1, k):
                cr[f"{columns[j]}_{columns[l]}"] = corr_mat[j, l]
        corr_rows.append(cr)

    df_var = pd.DataFrame(var_rows).set_index("date")
    df_corr = pd.DataFrame(corr_rows).set_index("date")
    return cov_dict, df_corr, df_var


def compute_betas_from_covariance(cov_dict, df_rets, target="r_pen",
                                   regressors=["r_cop", "r_clp", "r_mxn", "r_brl"],
                                   all_columns=["r_pen", "r_clp", "r_cop", "r_brl", "r_mxn"]):
    # Beta_i = Cov(PEN, i) / Var(i) from the EWMA covariance matrix
    target_idx = all_columns.index(target)
    reg_indices = [all_columns.index(r) for r in regressors]

    records = []
    for dt, sigma in cov_dict.items():
        betas = {}
        for r, ri in zip(regressors, reg_indices):
            var_i = sigma[ri, ri]
            cov_pen_i = sigma[target_idx, ri]
            betas["beta_" + r.split("_")[1]] = cov_pen_i / var_i if var_i > 0 else 0.0
        betas["date"] = dt
        records.append(betas)

    betas_df = pd.DataFrame(records).set_index("date")

    # Compute fitted and residual on the intersection
    common = betas_df.index.intersection(df_rets.index)
    fitted = pd.Series(0.0, index=common, dtype=float)
    for r in regressors:
        bcol = "beta_" + r.split("_")[1]
        fitted += betas_df.loc[common, bcol].values * df_rets.loc[common, r].values

    betas_df["fitted"] = np.nan
    betas_df["residual"] = np.nan
    betas_df.loc[common, "fitted"] = fitted.values
    betas_df.loc[common, "residual"] = (df_rets.loc[common, target] - fitted).values

    # Rolling 60-day R-squared
    actual = df_rets.loc[common, target]
    fit_series = pd.Series(fitted.values, index=common)
    resid_series = actual - fit_series
    roll_ss_res = (resid_series ** 2).rolling(60).sum()
    roll_ss_tot = ((actual - actual.rolling(60).mean()) ** 2).rolling(60).sum()
    r2 = 1 - roll_ss_res / roll_ss_tot
    betas_df["r_squared"] = np.nan
    betas_df.loc[common, "r_squared"] = r2.values

    return betas_df


def decompose_variance(betas_df, cov_dict, df_rets, decay=EWMA_LAMBDA, seed_window=SEED_WINDOW,
                        target="r_pen", regressors=["r_cop", "r_clp", "r_mxn", "r_brl"],
                        all_columns=["r_pen", "r_clp", "r_cop", "r_brl", "r_mxn"]):
    # Systematic = sum(beta_i * Cov(PEN, i)), idiosyncratic = EWMA of residual^2
    target_idx = all_columns.index(target)
    reg_indices = [all_columns.index(r) for r in regressors]

    residuals = betas_df["residual"].dropna()
    pen_rets = df_rets[target]

    # EWMA of residual^2
    resid_vals = residuals.values
    idio_ewma = np.var(resid_vals[:seed_window]) if len(resid_vals) >= seed_window else resid_vals[0] ** 2

    # EWMA of r_pen^2 for realized variance comparison
    common_dates = residuals.index.intersection(pen_rets.index)
    pen_vals = pen_rets.loc[common_dates].values
    real_ewma = np.var(pen_vals[:seed_window]) if len(pen_vals) >= seed_window else pen_vals[0] ** 2

    records = []
    for i, dt in enumerate(residuals.index):
        if dt not in cov_dict:
            continue

        sigma = cov_dict[dt]
        beta_cols = ["beta_" + r.split("_")[1] for r in regressors]
        beta_vec = betas_df.loc[dt, beta_cols].values.astype(float)

        # Systematic = sum(beta_i * Cov(PEN, i))
        sys_var = 0.0
        for j, ri in enumerate(reg_indices):
            sys_var += beta_vec[j] * sigma[target_idx, ri]

        # Update EWMA idiosyncratic variance
        if i > 0:
            idio_ewma = decay * idio_ewma + (1 - decay) * resid_vals[i - 1] ** 2

        # Update EWMA realized variance
        if dt in pen_rets.index:
            pi = list(common_dates).index(dt) if dt in common_dates else -1
            if pi > 0:
                real_ewma = decay * real_ewma + (1 - decay) * pen_vals[pi - 1] ** 2

        total = sys_var + idio_ewma
        records.append({
            "date": dt,
            "systematic_var": max(sys_var, 0),
            "idiosyncratic_var": idio_ewma,
            "total_var": total,
            "systematic_pct": sys_var / total if total > 0 else 0,
            "realized_var": real_ewma,
        })

    return pd.DataFrame(records).set_index("date")


def build_monitor(df_lvls, df_rets, betas_df, var_decomp_df, ewma_corr_df):
    # Merge everything into one daily monitoring DataFrame
    regressors = ["r_cop", "r_clp", "r_mxn", "r_brl"]
    common = betas_df.index.intersection(var_decomp_df.index).intersection(df_rets.index)

    records = []
    for dt in common:
        row = {"date": dt}
        row["usdpen_level"] = df_lvls.loc[dt, "usdpen"] if dt in df_lvls.index else np.nan
        row["usdpen_return"] = df_rets.loc[dt, "r_pen"]
        row["expected_return"] = betas_df.loc[dt, "fitted"] if "fitted" in betas_df.columns else np.nan
        row["residual"] = betas_df.loc[dt, "residual"] if "residual" in betas_df.columns else np.nan
        row["systematic_var"] = var_decomp_df.loc[dt, "systematic_var"]
        row["idiosyncratic_var"] = var_decomp_df.loc[dt, "idiosyncratic_var"]
        row["total_var"] = var_decomp_df.loc[dt, "total_var"]
        row["systematic_pct"] = var_decomp_df.loc[dt, "systematic_pct"]
        row["realized_var"] = var_decomp_df.loc[dt, "realized_var"]

        if dt in ewma_corr_df.index:
            row["avg_correlation"] = ewma_corr_df.loc[dt].mean()
        else:
            row["avg_correlation"] = np.nan

        beta_cols = ["beta_cop", "beta_clp", "beta_mxn", "beta_brl"]
        row["beta_sum"] = sum(betas_df.loc[dt, bc] for bc in beta_cols if bc in betas_df.columns)

        spot = row["usdpen_level"]
        idio = row["idiosyncratic_var"]
        if not np.isnan(spot) and idio > 0:
            row["range_95_pips"] = 2 * 1.96 * np.sqrt(idio) * spot / PIP
        else:
            row["range_95_pips"] = np.nan

        records.append(row)

    monitor = pd.DataFrame(records).set_index("date").sort_index()

    # Rolling 20-day cumulative residual and z-score
    monitor["cumul_residual_20d"] = monitor["residual"].rolling(RESID_ZSCORE_WINDOW).sum()
    roll_std = monitor["cumul_residual_20d"].rolling(RESID_ZSCORE_WINDOW).std()
    monitor["residual_zscore"] = monitor["cumul_residual_20d"] / roll_std

    return monitor


def conditional_range(regional_returns_dict, betas_dict, idio_var, spot, ci_levels=CI_LEVELS):
    # Compute conditional expected move and confidence bands
    expected_ret = sum(betas_dict.get("beta_" + k.split("_")[1], 0) * v
                       for k, v in regional_returns_dict.items())
    expected_pips = expected_ret * spot / PIP
    expected_level = spot * np.exp(expected_ret)
    idio_std = np.sqrt(idio_var)

    bands = {}
    for ci, z in ci_levels.items():
        half_pips = z * idio_std * spot / PIP
        lo = spot * np.exp(expected_ret - z * idio_std)
        hi = spot * np.exp(expected_ret + z * idio_std)
        bands[ci] = {"z": z, "half_pips": half_pips, "lower": lo, "upper": hi, "width_pips": 2 * half_pips}

    return {
        "expected_return": expected_ret,
        "expected_pips": expected_pips,
        "expected_level": expected_level,
        "idio_std": idio_std,
        "bands": bands,
    }

print("All model functions defined.")

## 4. Model Execution & Results

Run the model end-to-end and display key outputs.

### 4.1 — EWMA Covariance

In [ ]:
all_cols = list(df_returns.columns)  # ["r_pen", "r_clp", "r_cop", "r_brl", "r_mxn"]
cov_dict, df_corr, df_var = compute_ewma_covariance(df_returns, all_cols, decay=EWMA_LAMBDA)
print(f"Covariance matrix computed for {len(cov_dict)} dates, from {list(cov_dict.keys())[0].date()} to {list(cov_dict.keys())[-1].date()}.")

### 4.2 — Betas

In [ ]:
betas_df = compute_betas_from_covariance(cov_dict, df_returns, target="r_pen",
                                         regressors=["r_cop", "r_clp", "r_mxn", "r_brl"],
                                         all_columns=all_cols)
print("EWMA Betas (last 10):")
display(betas_df[["beta_cop", "beta_clp", "beta_mxn", "beta_brl", "r_squared"]].tail(10).round(4))

last_b = betas_df.iloc[-1]
print(f"\nCurrent betas -- COP: {last_b['beta_cop']:.3f}, CLP: {last_b['beta_clp']:.3f}, "
      f"MXN: {last_b['beta_mxn']:.3f}, BRL: {last_b['beta_brl']:.3f}. "
      f"R2: {last_b['r_squared']:.1%}")

### 4.3 — Variance Decomposition

In [ ]:
var_decomp = decompose_variance(betas_df, cov_dict, df_returns, decay=EWMA_LAMBDA,
                                all_columns=all_cols)
last_v = var_decomp.iloc[-1]
print(f"Current variance decomposition -- "
      f"Systematic: {last_v['systematic_pct']:.1%}, "
      f"Idiosyncratic: {1-last_v['systematic_pct']:.1%}. "
      f"Total daily vol: {np.sqrt(last_v['total_var'])*10000:.1f} bps. "
      f"Idio vol: {np.sqrt(last_v['idiosyncratic_var'])*10000:.1f} bps.")

### 4.4 — Build Monitor

In [ ]:
monitor = build_monitor(df_levels, df_returns, betas_df, var_decomp, df_corr)
monitor.to_csv("usdpen_monitor.csv")
print(f"Monitor saved: {len(monitor)} rows")
print("\nLast 20 rows:")
display(monitor.tail(20).round(4))

### 4.5 — Today's Range (Static Snapshot)

In [ ]:
# Unconditional range (zero regional input)
last_row = monitor.iloc[-1]
spot = last_row["usdpen_level"]
idio_v = last_row["idiosyncratic_var"]
sys_pct = last_row["systematic_pct"]
avg_corr = last_row["avg_correlation"]
z_score = last_row["residual_zscore"]
dt_str = str(monitor.index[-1].date())

cr0 = conditional_range({}, {}, idio_v, spot)

lines = []
lines.append("=" * 44)
lines.append(f"  USDPEN RANGE ESTIMATE -- {dt_str}")
lines.append("=" * 44)
lines.append(f"  Spot:           {spot:.4f}")
lines.append(f"  Expected move:  0 pips (no regional input)")
lines.append("")
for ci in [0.90, 0.95, 0.99]:
    b = cr0["bands"][ci]
    lines.append(f"  {ci:.0%} range:  {b['lower']:.4f} -- {b['upper']:.4f}  ({b['width_pips']:.0f} pips)")
lines.append("")
lines.append(f"  Systematic share: {sys_pct:.1%}")
lines.append(f"  Idio vol:         {np.sqrt(idio_v)*10000:.1f} bps/day")
lines.append(f"  Avg correlation:  {avg_corr:.3f}")
lines.append(f"  Residual z-score: {z_score:+.2f}")
lines.append("=" * 44)
print("\n".join(lines))

## 5. Live Update Widget

Input today's FX levels and instantly see the conditional range. The model uses the latest estimated betas and idiosyncratic variance — it does not re-estimate them (that would require the actual PEN close, which you don't have intraday).

### 5.1 — Widget

In [ ]:
from IPython.display import clear_output as _clear

last_lvls = df_levels.iloc[-1]
next_date = (df_levels.index[-1] + pd.tseries.offsets.BDay(1)).date()

w_date = widgets.Text(value=str(next_date), description="Date:", layout=widgets.Layout(width="200px"))
w_pen = widgets.FloatText(value=round(float(last_lvls["usdpen"]), 4), description="USDPEN:", step=0.0001,
                          layout=widgets.Layout(width="200px"))
w_cop = widgets.FloatText(value=round(float(last_lvls["usdcop"]), 2), description="USDCOP:", step=0.01,
                          layout=widgets.Layout(width="200px"))
w_clp = widgets.FloatText(value=round(float(last_lvls["usdclp"]), 2), description="USDCLP:", step=0.01,
                          layout=widgets.Layout(width="200px"))
w_mxn = widgets.FloatText(value=round(float(last_lvls["usdmxn"]), 4), description="USDMXN:", step=0.0001,
                          layout=widgets.Layout(width="200px"))
w_brl = widgets.FloatText(value=round(float(last_lvls["usdbrl"]), 4), description="USDBRL:", step=0.0001,
                          layout=widgets.Layout(width="200px"))

btn = widgets.Button(description="Compute Range", button_style="primary", icon="calculator")
out = widgets.Output()

def on_compute(b):
    with out:
        _clear(wait=True)

        prev = {"usdpen": float(last_lvls["usdpen"]), "usdcop": float(last_lvls["usdcop"]),
                "usdclp": float(last_lvls["usdclp"]), "usdmxn": float(last_lvls["usdmxn"]),
                "usdbrl": float(last_lvls["usdbrl"])}
        new = {"usdpen": w_pen.value, "usdcop": w_cop.value,
               "usdclp": w_clp.value, "usdmxn": w_mxn.value, "usdbrl": w_brl.value}

        reg_rets = {}
        for ccy in ["usdcop", "usdclp", "usdmxn", "usdbrl"]:
            rkey = "r_" + ccy.replace("usd", "")
            reg_rets[rkey] = np.log(new[ccy] / prev[ccy])

        lb = betas_df.iloc[-1]
        betas_dict = {"beta_cop": lb["beta_cop"], "beta_clp": lb["beta_clp"],
                      "beta_mxn": lb["beta_mxn"], "beta_brl": lb["beta_brl"]}
        idio_v_w = var_decomp["idiosyncratic_var"].iloc[-1]
        sys_v_w = var_decomp["systematic_var"].iloc[-1]
        spot_w = new["usdpen"]

        cr = conditional_range(reg_rets, betas_dict, idio_v_w, spot_w)

        # Contributions
        contribs = {}
        for rkey in ["r_cop", "r_clp", "r_mxn", "r_brl"]:
            bk = "beta_" + rkey.split("_")[1]
            contribs[rkey] = betas_dict[bk] * reg_rets[rkey] * spot_w / PIP

        total_v = sys_v_w + idio_v_w
        z_sc = monitor["residual_zscore"].iloc[-1] if "residual_zscore" in monitor.columns else 0
        signal = "PEN RICH" if z_sc < -1.5 else ("PEN CHEAP" if z_sc > 1.5 else "FAIR")

        ccy_display = {"usdcop": "USDCOP", "usdclp": "USDCLP", "usdmxn": "USDMXN", "usdbrl": "USDBRL"}
        L = []
        L.append("=" * 64)
        L.append(f"  USDPEN CONDITIONAL RANGE -- {w_date.value}")
        L.append("=" * 64)
        L.append("  REGIONAL MOVES TODAY")
        for ccy in ["usdcop", "usdclp", "usdmxn", "usdbrl"]:
            rk = "r_" + ccy.replace("usd", "")
            pips_chg = (new[ccy] - prev[ccy]) / PIP
            pct_chg = reg_rets[rk] * 100
            L.append(f"    {ccy_display[ccy]}:  {new[ccy]:.4f}  ({pips_chg:+.0f} pips  / {pct_chg:+.2f}%)")
        L.append("")
        L.append("  MODEL OUTPUT FOR USDPEN")
        L.append(f"    Yesterday's close:     {prev['usdpen']:.4f}")
        L.append(f"    Expected move:         {cr['expected_pips']:+.1f} pips  ({cr['expected_return']*10000:+.1f} bps)")
        L.append(f"    Expected level:        {cr['expected_level']:.4f}")
        L.append("")
        for ci in [0.90, 0.95, 0.99]:
            bd = cr["bands"][ci]
            L.append(f"    {ci:.0%} band:  {bd['lower']:.4f} -- {bd['upper']:.4f}   (+/-{bd['half_pips']:.0f} pips)")
        L.append("")
        L.append("  BREAKDOWN (who's moving PEN today)")
        for rk in ["r_cop", "r_clp", "r_mxn", "r_brl"]:
            bk = "beta_" + rk.split("_")[1]
            label = rk.split("_")[1].upper()
            L.append(f"    {label} contribution:  {contribs[rk]:+.1f} pips  (beta={betas_dict[bk]:.3f})")
        L.append("")
        L.append("  RISK CONTEXT")
        L.append(f"    Systematic var share:  {sys_v_w/total_v:.1%}")
        L.append(f"    Idio daily vol:        {np.sqrt(idio_v_w)*10000:.1f} bps")
        L.append(f"    Avg LatAm correlation: {monitor['avg_correlation'].iloc[-1]:.2f}")
        L.append(f"    Residual z-score (20d): {z_sc:+.2f}  [{signal}]")
        L.append("=" * 64)
        print("\n".join(L))

        # Contribution bar chart
        fig, ax = plt.subplots(figsize=(8, 2.5))
        labels_c = [rk.split("_")[1].upper() for rk in contribs]
        vals_c = list(contribs.values())
        cmap = {"COP": COLORS["cop"], "CLP": COLORS["clp"], "MXN": COLORS["mxn"], "BRL": COLORS["brl"]}
        bar_colors = [cmap.get(l, "gray") for l in labels_c]
        ax.barh(labels_c, vals_c, color=bar_colors)
        ax.axvline(0, color="black", linewidth=0.5)
        ax.set_xlabel("Contribution (pips)")
        ax.set_title("Contribution to Expected USDPEN Move (pips)")
        plt.tight_layout()
        plt.show()

btn.on_click(on_compute)

row1 = widgets.HBox([w_date, w_pen, w_cop])
row2 = widgets.HBox([w_clp, w_mxn, w_brl])
display(widgets.VBox([row1, row2, btn, out]))

## 6. Charts

Visual diagnostics. All charts use regime shading: light gray = BCRP intervention (Nov25–Feb26), light red = Iran conflict (Mar26+).

### 6.1 — Betas Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for bcol, ck in [("beta_cop", "cop"), ("beta_clp", "clp"), ("beta_mxn", "mxn"), ("beta_brl", "brl")]:
    ax.plot(betas_df.index, betas_df[bcol], label=bcol.replace("beta_", "").upper(),
            color=COLORS[ck], linewidth=1.2)
shade_regimes(ax)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("USDPEN Betas on Regional Currencies")
ax.set_ylabel("Beta")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 6.2 — Variance Decomposition

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
vd = var_decomp.copy()
ax.fill_between(vd.index, 0, vd["systematic_var"] * 1e8, alpha=0.7, color=COLOR_SYS, label="Systematic")
ax.fill_between(vd.index, vd["systematic_var"] * 1e8, vd["total_var"] * 1e8,
                alpha=0.7, color=COLOR_IDIO, label="Idiosyncratic")
ax.plot(vd.index, vd["realized_var"] * 1e8, color=COLOR_TOTAL, linestyle="--", linewidth=1.2,
        label="Realized EWMA Var")
shade_regimes(ax)
ax.set_title("USDPEN Variance Decomposition")
ax.set_ylabel("Variance (x1e-8)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 6.3 — Systematic Share

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(var_decomp.index, var_decomp["systematic_pct"] * 100, color=COLOR_SYS, linewidth=1.2)
ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, label="50%")
ax.axhline(75, color="gray", linestyle=":", linewidth=0.8, label="75%")
shade_regimes(ax)
ax.set_title("Regional Share of USDPEN Variance")
ax.set_ylabel("Systematic %")
ax.set_ylim(0, 100)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 6.4 — Average Pairwise Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_corr.index, df_corr.mean(axis=1), color=COLOR_SYS, linewidth=1.2)
shade_regimes(ax)
ax.set_title("Average Pairwise LatAm FX Correlation")
ax.set_ylabel("Avg Correlation")
plt.tight_layout()
plt.show()

### 6.5 — Conditional Range: Last 3 Months

In [ ]:
def plot_cond_range(mon_df, title, n_days=None):
    df_p = mon_df.dropna(subset=["range_95_pips", "usdpen_return"]).copy()
    if n_days:
        df_p = df_p.tail(n_days)

    spot_p = df_p["usdpen_level"]
    actual_pips = df_p["usdpen_return"] * spot_p / PIP
    exp_pips = df_p["expected_return"] * spot_p / PIP
    idio_std_p = np.sqrt(df_p["idiosyncratic_var"])

    fig, ax = plt.subplots(figsize=(14, 5))
    for ci, z, alpha_v, lbl in [(0.99, 2.5758, 0.12, "99% CI"), (0.95, 1.96, 0.25, "95% CI")]:
        hi = exp_pips + z * idio_std_p * spot_p / PIP
        lo = exp_pips - z * idio_std_p * spot_p / PIP
        ax.fill_between(df_p.index, lo, hi, alpha=alpha_v, color=COLOR_SYS, label=lbl)

    ax.plot(df_p.index, exp_pips, color=COLOR_TOTAL, linewidth=1, label="Expected")
    ax.scatter(df_p.index, actual_pips, s=10, color=COLOR_TOTAL, alpha=0.5, label="Actual", zorder=5)
    shade_regimes(ax)
    ax.set_title(title)
    ax.set_ylabel("Pips")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

plot_cond_range(monitor, "USDPEN Daily Moves vs Model Bands -- Last 3 Months", n_days=63)

### 6.6 — Conditional Range: Full Sample

In [ ]:
plot_cond_range(monitor, "USDPEN Daily Moves vs Model Bands -- Full Sample")

### 6.7 — Residual Z-Score

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
z_vals = monitor["residual_zscore"].dropna()
colors_z = ["#2563EB" if v < -1.5 else "#DC2626" if v > 1.5 else "#9CA3AF" for v in z_vals]
ax.scatter(z_vals.index, z_vals.values, c=colors_z, s=10, alpha=0.7)
ax.axhline(1.5, color="#DC2626", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(-1.5, color="#2563EB", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(2.0, color="#DC2626", linestyle=":", linewidth=0.8, alpha=0.3)
ax.axhline(-2.0, color="#2563EB", linestyle=":", linewidth=0.8, alpha=0.3)
ax.axhspan(-1.5, 1.5, alpha=0.04, color="gray")
shade_regimes(ax)
ax.set_title("USDPEN Idiosyncratic Residual Z-Score")
ax.set_ylabel("Z-Score")
plt.tight_layout()
plt.show()

### 6.8 — Beta Evolution Heatmap

In [ ]:
bm = betas_df[["beta_cop", "beta_clp", "beta_mxn", "beta_brl"]].copy()
bm["month"] = bm.index.to_period("M")
heat = bm.groupby("month").last().T  # month-end values

fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(heat.values, aspect="auto", cmap="RdYlBu_r")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels([b.replace("beta_", "").upper() for b in heat.index])
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels([str(p) for p in heat.columns], rotation=45, ha="right", fontsize=8)
plt.colorbar(im, ax=ax, label="Beta")
ax.set_title("USDPEN Beta Evolution")
plt.tight_layout()
plt.show()

## 7. Backtesting

The model claims that 95% of moves should fall inside the 95% band. We test whether that's true out of sample:
- **Coverage**: does the empirical breach rate match the theoretical rate?
- **Independence**: do breaches cluster (a breach today predicts a breach tomorrow)?
- **Calibration**: is the full conditional distribution correct, not just the tails?

### 7.1 — Backtest Functions

In [ ]:
def backtest_coverage(mon_df, train_window=120):
    # Walk-forward coverage test
    df = mon_df.dropna(subset=["usdpen_return", "expected_return", "idiosyncratic_var"]).copy()
    if len(df) <= train_window:
        print("Not enough data")
        return None
    test = df.iloc[train_window:]
    actual = test["usdpen_return"].values
    expected = test["expected_return"].values
    idio_std = np.sqrt(test["idiosyncratic_var"].values)

    results = {"n_test": len(test), "test_dates": test.index}

    for ci, z in CI_LEVELS.items():
        upper = expected + z * idio_std
        lower = expected - z * idio_std
        breaches = (actual > upper) | (actual < lower)
        n1 = breaches.sum()
        n0 = len(breaches) - n1
        n_total = len(breaches)
        emp_rate = n1 / n_total
        theo_rate = 1 - ci

        # Christoffersen unconditional LR
        p0 = theo_rate
        p1 = max(min(emp_rate, 1 - 1e-10), 1e-10)
        log_l0 = n0 * np.log(1 - p0) + n1 * np.log(p0)
        log_l1 = n0 * np.log(1 - p1) + n1 * np.log(p1)
        lr_uc = max(-2 * (log_l0 - log_l1), 0)
        pval_uc = 1 - stats.chi2.cdf(lr_uc, 1)

        # Independence LR
        b = breaches.astype(int)
        t00 = t01 = t10 = t11 = 0
        for k in range(1, len(b)):
            if b[k-1] == 0 and b[k] == 0: t00 += 1
            elif b[k-1] == 0 and b[k] == 1: t01 += 1
            elif b[k-1] == 1 and b[k] == 0: t10 += 1
            else: t11 += 1

        eps = 1e-10
        pi01 = max(t01 / (t00 + t01), eps) if (t00 + t01) > 0 else eps
        pi11 = max(t11 / (t10 + t11), eps) if (t10 + t11) > 0 else eps
        pi = max((t01 + t11) / n_total, eps)

        log_l1_ind = (t00 * np.log(1 - pi01 + eps) + t01 * np.log(pi01 + eps))
        if (t10 + t11) > 0:
            log_l1_ind += t10 * np.log(1 - pi11 + eps) + t11 * np.log(pi11 + eps)
        log_l0_ind = (t00 + t10) * np.log(1 - pi + eps) + (t01 + t11) * np.log(pi + eps)
        lr_ind = max(2 * (log_l1_ind - log_l0_ind), 0)
        pval_ind = 1 - stats.chi2.cdf(lr_ind, 1)

        lr_cc = lr_uc + lr_ind
        pval_cc = 1 - stats.chi2.cdf(lr_cc, 2)

        results[ci] = {
            "theo_rate": theo_rate, "emp_rate": emp_rate,
            "n_breaches": int(n1), "n_total": n_total,
            "lr_uc": lr_uc, "pval_uc": pval_uc,
            "lr_ind": lr_ind, "pval_ind": pval_ind,
            "lr_cc": lr_cc, "pval_cc": pval_cc,
            "breach_mask": breaches,
            "breach_dates": test.index[breaches],
        }

    return results


def pit_test(mon_df, train_window=120):
    # Probability Integral Transform test
    df = mon_df.dropna(subset=["usdpen_return", "expected_return", "idiosyncratic_var"]).copy()
    test = df.iloc[train_window:]
    actual = test["usdpen_return"].values
    expected = test["expected_return"].values
    idio_std = np.sqrt(test["idiosyncratic_var"].values)
    z_scores = (actual - expected) / idio_std
    pit_vals = stats.norm.cdf(z_scores)
    ks_stat, ks_pval = stats.kstest(pit_vals, "uniform")
    return {"ks_stat": ks_stat, "ks_pval": ks_pval, "pit_values": pit_vals}


def basel_traffic_light(mon_df, ci_level=0.99, window=250, train_window=120):
    # Rolling count of breaches at 99%
    df = mon_df.dropna(subset=["usdpen_return", "expected_return", "idiosyncratic_var"]).copy()
    test = df.iloc[train_window:]
    actual = test["usdpen_return"].values
    expected = test["expected_return"].values
    idio_std = np.sqrt(test["idiosyncratic_var"].values)
    z = CI_LEVELS[ci_level]
    breaches = ((actual > expected + z * idio_std) | (actual < expected - z * idio_std)).astype(int)
    breach_series = pd.Series(breaches, index=test.index)
    effective_window = min(window, len(breach_series))
    rolling_count = breach_series.rolling(effective_window, min_periods=1).sum()
    colors_tl = rolling_count.apply(lambda x: "Green" if x <= 4 else ("Yellow" if x <= 9 else "Red"))
    return rolling_count, colors_tl, effective_window


def mm_backtest(mon_df, ci_level=0.95, holding_days=1, cost_pips=3, train_window=120):
    # Market-making fade strategy
    df = mon_df.dropna(subset=["usdpen_return", "expected_return", "idiosyncratic_var", "usdpen_level"]).copy()
    test = df.iloc[train_window:]
    z = CI_LEVELS[ci_level]
    trades = []

    for i in range(len(test)):
        row = test.iloc[i]
        actual = row["usdpen_return"]
        exp = row["expected_return"]
        idio_std = np.sqrt(row["idiosyncratic_var"])
        spot_mm = row["usdpen_level"]
        upper = exp + z * idio_std
        lower = exp - z * idio_std

        if actual > upper or actual < lower:
            direction = "sell" if actual > upper else "buy"
            if i + holding_days < len(test):
                next_ret = test.iloc[i + holding_days]["usdpen_return"]
                if direction == "sell":
                    pnl = -next_ret * spot_mm / PIP - cost_pips
                else:
                    pnl = next_ret * spot_mm / PIP - cost_pips
                trades.append({"date": test.index[i], "direction": direction,
                              "pnl_pips": pnl, "regime": label_regime(test.index[i])})

    if not trades:
        return {"total_trades": 0, "trade_df": pd.DataFrame()}

    tdf = pd.DataFrame(trades).set_index("date")
    tdf["cumul_pnl"] = tdf["pnl_pips"].cumsum()
    n_trades = len(tdf)
    win_rate = (tdf["pnl_pips"] > 0).mean()
    avg_pnl = tdf["pnl_pips"].mean()
    sharpe = avg_pnl / tdf["pnl_pips"].std() * np.sqrt(252) if tdf["pnl_pips"].std() > 0 else 0
    max_dd = (tdf["cumul_pnl"] - tdf["cumul_pnl"].cummax()).min()

    return {"total_trades": n_trades, "win_rate": win_rate, "avg_pnl": avg_pnl,
            "sharpe": sharpe, "max_dd": max_dd, "trade_df": tdf}

print("Backtest functions defined.")

### 7.2 — Run Coverage Backtest

In [ ]:
cov_res = backtest_coverage(monitor, train_window=120)

if cov_res:
    n = cov_res["n_test"]
    print(f"Coverage Backtest Results (train window: 120 days, test: {n} days)")
    print("-" * 75)
    print(f"{'CI':>6} | {'Theoretical':>11} | {'Empirical':>10} | {'Breaches':>8} | {'LR_uc':>8} | {'p-value':>8}")
    print("-" * 75)
    for ci in [0.90, 0.95, 0.99]:
        r = cov_res[ci]
        print(f"{ci:6.0%} | {r['theo_rate']:11.1%} | {r['emp_rate']:10.1%} | "
              f"{r['n_breaches']:8d} | {r['lr_uc']:8.3f} | {r['pval_uc']:8.3f}")
    print("-" * 75)

    # Independence and conditional coverage for 95%
    r95 = cov_res[0.95]
    print(f"\nIndependence test (95%):      LR = {r95['lr_ind']:.3f}, p = {r95['pval_ind']:.3f}")
    print(f"Conditional coverage (95%):  LR = {r95['lr_cc']:.3f}, p = {r95['pval_cc']:.3f}")
    verdict = "PASS" if r95["pval_cc"] > 0.05 else "FAIL"
    print(f"Interpretation: {verdict} at 5% significance")

### 7.3 — PIT Test

In [ ]:
pit_res = pit_test(monitor, train_window=120)
print(f"PIT Test: KS stat = {pit_res['ks_stat']:.4f}, p-value = {pit_res['ks_pval']:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pit_res["pit_values"], bins=20, range=(0, 1), density=True,
        alpha=0.7, color=COLOR_SYS, edgecolor="white")
ax.axhline(1.0, color="red", linestyle="--", linewidth=1, label="Uniform (ideal)")
ax.set_xlabel("PIT Value")
ax.set_ylabel("Density")
ax.set_title("PIT Histogram -- Model Calibration")
ax.legend()
plt.tight_layout()
plt.show()

### 7.4 — Coverage Bar Chart

In [ ]:
if cov_res:
    fig, ax = plt.subplots(figsize=(8, 4))
    ci_labels = ["90%", "95%", "99%"]
    theo = [cov_res[ci]["theo_rate"] * 100 for ci in [0.90, 0.95, 0.99]]
    emp = [cov_res[ci]["emp_rate"] * 100 for ci in [0.90, 0.95, 0.99]]
    x = np.arange(3)
    w = 0.35
    ax.bar(x - w/2, theo, w, label="Theoretical", color=COLOR_SYS, alpha=0.8)
    ax.bar(x + w/2, emp, w, label="Empirical", color=COLOR_IDIO, alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(ci_labels)
    ax.set_ylabel("Breach Rate (%)")
    ax.set_title("Backtest: Theoretical vs Empirical Breach Rates")
    ax.legend()
    plt.tight_layout()
    plt.show()

### 7.5 — Breach Timeline

In [ ]:
if cov_res and 0.99 in cov_res:
    fig, ax = plt.subplots(figsize=(14, 4))
    spot_bt = monitor["usdpen_level"]
    actual_pips_bt = monitor["usdpen_return"] * spot_bt / PIP
    ax.plot(monitor.index, actual_pips_bt, color="gray", alpha=0.4, linewidth=0.8)

    regime_colors = {"normal": "#6B7280", "intervention": "#3B82F6", "post_shock": "#EF4444"}
    for dt in cov_res[0.99]["breach_dates"]:
        if dt in monitor.index:
            regime = label_regime(dt)
            ax.scatter(dt, actual_pips_bt.loc[dt], color=regime_colors[regime], s=40,
                      zorder=5, edgecolors="black", linewidth=0.5)

    for regime, color in regime_colors.items():
        ax.scatter([], [], color=color, label=regime, s=40, edgecolors="black", linewidth=0.5)
    shade_regimes(ax)
    ax.set_title("99% CI Breaches Over Time")
    ax.set_ylabel("Pips")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

### 7.6 — Basel Traffic Light

In [ ]:
rolling_br, tl_colors, eff_window = basel_traffic_light(monitor)

if eff_window < 250:
    print(f"Note: Dataset shorter than 250 days. Using full available window ({eff_window} days). Results are preliminary.")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rolling_br.index, rolling_br.values, color=COLOR_TOTAL, linewidth=1.2)

# Background coloring
color_map = {"Green": "#10B981", "Yellow": "#F59E0B", "Red": "#EF4444"}
prev_dt = rolling_br.index[0]
for i in range(1, len(rolling_br)):
    c = tl_colors.iloc[i]
    ax.axvspan(rolling_br.index[i-1], rolling_br.index[i], alpha=0.12, color=color_map.get(c, "gray"))

ax.axhline(4, color="#F59E0B", linestyle="--", linewidth=0.8, label="Green/Yellow (4)")
ax.axhline(9, color="#EF4444", linestyle="--", linewidth=0.8, label="Yellow/Red (9)")
ax.set_title(f"Basel Traffic Light -- Rolling {eff_window}-Day 99% Breaches")
ax.set_ylabel("Breach Count")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

print(f"Current status: {tl_colors.iloc[-1]} ({int(rolling_br.iloc[-1])} breaches)")

### 7.7 — Market-Making Simulation

In [ ]:
mm_res = mm_backtest(monitor, ci_level=0.95, holding_days=1, cost_pips=3)

if mm_res["total_trades"] > 0:
    print("Fade Strategy Backtest (95% CI, 1-day hold, 3 pip cost)")
    print("-" * 55)
    print(f"  Total trades:    {mm_res['total_trades']}")
    print(f"  Win rate:        {mm_res['win_rate']:.1%}")
    print(f"  Avg P&L/trade:   {mm_res['avg_pnl']:.1f} pips")
    print(f"  Sharpe (ann):    {mm_res['sharpe']:.2f}")
    print(f"  Max drawdown:    {mm_res['max_dd']:.1f} pips")
    print("-" * 55)

    tdf = mm_res["trade_df"]
    print("\nBy regime:")
    for regime in ["normal", "intervention", "post_shock"]:
        sub = tdf[tdf["regime"] == regime]
        if len(sub) > 0:
            print(f"  {regime}: {len(sub)} trades, avg P&L = {sub['pnl_pips'].mean():.1f} pips, total = {sub['pnl_pips'].sum():.1f} pips")

    fig, ax = plt.subplots(figsize=(14, 4))
    regime_colors_mm = {"normal": "#6B7280", "intervention": "#3B82F6", "post_shock": "#EF4444"}
    for regime in ["normal", "intervention", "post_shock"]:
        mask = tdf["regime"] == regime
        if mask.any():
            sub = tdf[mask]
            ax.fill_between(sub.index, 0, sub["cumul_pnl"], alpha=0.3,
                          color=regime_colors_mm[regime], label=regime)
    ax.plot(tdf.index, tdf["cumul_pnl"], color=COLOR_TOTAL, linewidth=1.2)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title("Fade Strategy -- Cumulative P&L (pips)")
    ax.set_ylabel("Cumulative P&L (pips)")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No trades generated.")